In [1]:
# Import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
# Load the datasets
df1 = pd.read_csv('spam.csv', encoding='latin-1')
df2 = pd.read_csv('SMS Spam Dataset.csv', encoding='latin-1')

# Preview both datasets
print("Dataset 1 — spam.csv:")
print(df1.head())
print("\nShape:", df1.shape)
print("\n---")
print("Dataset 2 — SMS Spam Dataset.csv:")
print(df2.head())
print("\nShape:", df2.shape)

Dataset 1 — spam.csv:
     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  

Shape: (5572, 5)

---
Dataset 2 — SMS Spam Dataset.csv:
                                                 sms  label
0  Go until jurong point, crazy.. Available only ...      0
1                    Ok lar... Joking wif u oni...\n      0
2  Free entry in 2 a wkly comp to win FA Cup fina...      1
3  U dun say so early hor... U c already then say...      0
4  Nah I don't think he go

In [3]:
# Clean Dataset 1
df1 = df1[['v1', 'v2']]
df1.columns = ['label', 'message']
df1['label'] = df1['label'].map({'ham': 0, 'spam': 1})

# Clean Dataset 2
df2.columns = ['message', 'label']
df2 = df2[['label', 'message']]

# Combine both datasets
df = pd.concat([df1, df2], ignore_index=True)

# Remove duplicates
df = df.drop_duplicates()

# Remove any empty rows
df = df.dropna()

# Check the result
print("Combined Dataset:")
print(df.head(10))
print("\nShape:", df.shape)
print("\nLabel distribution:")
print(df['label'].value_counts())

Combined Dataset:
   label                                            message
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...
5      1  FreeMsg Hey there darling it's been 3 week's n...
6      0  Even my brother is not like to speak with me. ...
7      0  As per your request 'Melle Melle (Oru Minnamin...
8      1  WINNER!! As a valued network customer you have...
9      1  Had your mobile 11 months or more? U R entitle...

Shape: (10340, 2)

Label distribution:
0    9034
1    1306
Name: label, dtype: int64


In [4]:
# Add scam-specific features
import re

def count_urgency_words(text):
    urgency_words = ['urgent', 'immediately', 'now', 'hurry', 'quick', 
                     'fast', 'limited', 'expire', 'deadline', 'asap',
                     'don\'t wait', 'act now', 'before it\'s too late']
    text = text.lower()
    return sum(1 for word in urgency_words if word in text)

def count_pressure_phrases(text):
    pressure_phrases = ['trust me', 'don\'t worry', 'i also did it',
                        'just ignore', 'it\'s fine', 'believe me',
                        'i promise', 'guaranteed', 'no risk']
    text = text.lower()
    return sum(1 for phrase in pressure_phrases if phrase in text)

def count_money_requests(text):
    money_words = ['send money', 'transfer', 'bank account', 'payment',
                   'crypto', 'bitcoin', 'gift card', 'wire', 'cash',
                   'invest', 'profit', 'earn', 'winning', 'prize']
    text = text.lower()
    return sum(1 for word in money_words if word in text)

def count_suspicious_links(text):
    url_pattern = re.compile(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+])+')
    urls = url_pattern.findall(text)
    return len(urls)

def count_secrecy_phrases(text):
    secrecy_words = ['don\'t tell', 'keep secret', 'between us',
                     'don\'t share', 'confidential', 'private',
                     'don\'t show', 'just you and me']
    text = text.lower()
    return sum(1 for phrase in secrecy_words if phrase in text)

# Apply all features to the dataset
df['urgency_count'] = df['message'].apply(count_urgency_words)
df['pressure_count'] = df['message'].apply(count_pressure_phrases)
df['money_request_count'] = df['message'].apply(count_money_requests)
df['suspicious_links'] = df['message'].apply(count_suspicious_links)
df['secrecy_count'] = df['message'].apply(count_secrecy_phrases)
df['message_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(x.split()))

print("Features added successfully!")
print(df.head())
print("\nFeature summary for scam messages:")
print(df[df['label']==1][['urgency_count', 'pressure_count', 
                            'money_request_count', 'suspicious_links',
                            'secrecy_count']].describe())

Features added successfully!
   label                                            message  urgency_count  \
0      0  Go until jurong point, crazy.. Available only ...              0   
1      0                      Ok lar... Joking wif u oni...              0   
2      1  Free entry in 2 a wkly comp to win FA Cup fina...              0   
3      0  U dun say so early hor... U c already then say...              0   
4      0  Nah I don't think he goes to usf, he lives aro...              0   

   pressure_count  money_request_count  suspicious_links  secrecy_count  \
0               0                    0                 0              0   
1               0                    0                 0              0   
2               0                    0                 0              0   
3               0                    0                 0              0   
4               0                    0                 0              0   

   message_length  word_count  
0             111  

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import scipy.sparse as sp

# Step 1 — TF-IDF on the message text
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = tfidf.fit_transform(df['message'])

# Step 2 — Our custom scam features
custom_features = df[['urgency_count', 'pressure_count', 
                       'money_request_count', 'suspicious_links',
                       'secrecy_count', 'message_length', 
                       'word_count']].values

# Step 3 — Combine TF-IDF + custom features together
X = sp.hstack([X_tfidf, custom_features])

# Step 4 — Labels
y = df['label']

# Step 5 — Split into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Data preparation complete!")
print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")
print(f"Scam messages in training: {sum(y_train==1)}")
print(f"Legitimate messages in training: {sum(y_train==0)}")

Data preparation complete!
Training set size: (8272, 5007)
Testing set size: (2068, 5007)
Scam messages in training: 1045
Legitimate messages in training: 7227


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Train Logistic Regression model
print("Training model...")
model = LogisticRegression(
    class_weight='balanced',  # handles imbalanced dataset
    max_iter=1000,
    random_state=42
)
model.fit(X_train, y_train)

# Test the model
y_pred = model.predict(X_test)

# Results
print("\n Model trained successfully!")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, 
      target_names=['Legitimate', 'Scam']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Training model...

 Model trained successfully!

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      0.99      0.99      1807
        Scam       0.91      0.98      0.95       261

    accuracy                           0.99      2068
   macro avg       0.96      0.98      0.97      2068
weighted avg       0.99      0.99      0.99      2068


Confusion Matrix:
[[1783   24]
 [   5  256]]


In [7]:
def analyse_message(message):
    # Extract custom features
    urgency = count_urgency_words(message)
    pressure = count_pressure_phrases(message)
    money = count_money_requests(message)
    links = count_suspicious_links(message)
    secrecy = count_secrecy_phrases(message)
    length = len(message)
    words = len(message.split())
    
    # TF-IDF transform
    message_tfidf = tfidf.transform([message])
    
    # Combine features
    custom = [[urgency, pressure, money, links, secrecy, length, words]]
    message_features = sp.hstack([message_tfidf, custom])
    
    # Get prediction and probability
    prediction = model.predict(message_features)[0]
    probability = model.predict_proba(message_features)[0][1]
    
    # Calculate risk score (0-100)
    risk_score = int(probability * 100)
    
    # Determine risk level
    if risk_score >= 70:
        risk_level = "High"
    elif risk_score >= 40:
        risk_level = "Medium"
    else:
        risk_level = "Low"
    
    # Determine triggered features
    triggered_features = []
    if urgency > 0:
        triggered_features.append("urgency")
    if pressure > 0:
        triggered_features.append("pressure_tactics")
    if money > 0:
        triggered_features.append("money_request")
    if links > 0:
        triggered_features.append("suspicious_links")
    if secrecy > 0:
        triggered_features.append("secrecy_tactics")
    if not triggered_features and prediction == 1:
        triggered_features.append("suspicious_language_pattern")
    
    return {
        "risk_score": risk_score,
        "risk_level": risk_level,
        "confidence": round(float(probability), 2),
        "triggered_features": triggered_features,
        "requires_human_review": risk_level in ["Medium", "High"]
    }

# Test with some examples
print("Test 1 — Scam message:")
result1 = analyse_message("URGENT! Send money now to my account. Trust me don't worry. Limited time offer!")
print(result1)

print("\nTest 2 — Legitimate message:")
result2 = analyse_message("Hey, are we still meeting for lunch tomorrow?")
print(result2)

print("\nTest 3 — Social media impersonation scam:")
result3 = analyse_message("Hi it's me, I need you to transfer money urgently to this account, don't tell anyone please")
print(result3)

Test 1 — Scam message:
{'risk_score': 96, 'risk_level': 'High', 'confidence': 0.96, 'triggered_features': ['urgency', 'pressure_tactics', 'money_request'], 'requires_human_review': True}

Test 2 — Legitimate message:
{'risk_score': 2, 'risk_level': 'Low', 'confidence': 0.02, 'triggered_features': [], 'requires_human_review': False}

Test 3 — Social media impersonation scam:
{'risk_score': 64, 'risk_level': 'Medium', 'confidence': 0.65, 'triggered_features': ['urgency', 'money_request', 'secrecy_tactics'], 'requires_human_review': True}


In [9]:
import pickle

# Save the trained model
with open('scamshield_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save the TF-IDF vectorizer
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print(" Model saved as scamshield_model.pkl")
print(" TF-IDF vectorizer saved as tfidf_vectorizer.pkl")
print("\nThese files can now be loaded by the backend API!")

 Model saved as scamshield_model.pkl
 TF-IDF vectorizer saved as tfidf_vectorizer.pkl

These files can now be loaded by the backend API!
